## Lab 3 Exercises (Due: 9/18/26)
In this lab we will become familiar with distributions, histograms, and functional programming. Do not use numpy or any other library for this lab.

### Uniform Distribution
Lets start with generating some fake random data. You can get a random number between 0 and 1 using the python random module as follow:

In [ ]:
import random
x=random.random()
print("The Value of x is", x)

Everytime you call random, you will get a new number.

*Exercise 1:* Using random, write a function `generate_uniform(N, mymin, mymax)`, that returns a python list containing N random numbers between specified minimum and maximum value. Note that you may want to quickly work out on paper how to turn numbers between 0 and 1 to between other values. 

In [ ]:
def generate_uniform(N, x_min, x_max):
    out = []
    for i in range(N):
        x = random.random()
        out.append(x_min + x * (x_max - x_min))
    return out


In [ ]:
data=generate_uniform(1000,-10,10)
print ("Data Type:", type(data))
print ("Data Length:", len(data))
if len(data)>0: 
    print ("Type of Data Contents:", type(data[0]))
    print ("Data Minimum:", min(data))
    print ("Data Maximum:", max(data))

*Exercise 2a:* 
Write a function that computes the mean of values in a list. Recall the equation for the mean of a random variable $\bf{x}$ computed on a data set of $n$ values $\{ x_i \} = \{x_1, x_2, ..., x_n\}$  is ${\bf\bar{x}} = \frac{1}{n} \sum_i^n x_i$.

In [1]:
def mean(Data):
    if len(Data) == 0:
        return 0.0
    m = 0.0
    for x in Data:
        m += x
    m /= len(Data)
    return m


In [ ]:
print ("Mean of Data:", mean(data))

*Exercise 2b:* 
Write a function that computes the variance of values in a list. Recall the equation for the variance of a random variable $\bf{x}$ computed on a data set of $n$ values $\{ x_i \} = \{x_1, x_2, ..., x_n\}$  is $s^2 = \frac{1}{n-1} \sum_i^n (x_i - {\bf\bar{x}})^2$.

(Note the notation: $\langle x \rangle$ means the *mean*, so the variance is written $s^2$ — or
$\sigma^2$ when it is the variance of a distribution rather than of a sample. The $n-1$ rather
than $n$ is the Bessel correction: you had to estimate $\bf\bar{x}$ from the same data, which
spends one degree of freedom.)

In [ ]:
def variance(Data):
    if len(Data) < 2:
        return 0.0
    m = mean(Data)
    total = 0.0
    for x in Data:
        total += (x - m) ** 2
    return total / (len(Data) - 1)


In [ ]:
# Test your solution here
print ("Variance of Data:", variance(data))

## Histogramming

*Exercise 3:* Write a function that bins the data so that you can create a histogram. An example of how to implement histogramming is the following logic:

* User inputs a list of values `x` and optionally `n_bins` which defaults to 10.
* If not supplied, find the minimum and maximum (`x_min`,`x_max`) of the values in x.
* Determine the bin size (`bin_size`) by dividing the range of the function by the number of bins.
* Create an empty list of zeros of size `n_bins`, call it `hist`.
* Loop over the values in `x`
    * Loop over the values in `hist` with index `i`:
        * If x is between `x_min+i*bin_size` and `x_min+(i+1)*bin_size`, increment `hist[i].` 
        * Once you have found the right bin there is no reason to keep checking the rest, so use `break` to stop the inner loop and move on to the next data point.
* Return `hist` and the list corresponding of the bin edges (i.e. of `x_min+i*bin_size`).    

In [ ]:
def histogram(x, n_bins=10, x_min=None, x_max=None):
    if len(x) == 0:
        return [0] * n_bins, []
    if x_min is None:
        x_min = min(x)
    if x_max is None:
        x_max = max(x)
    if x_max == x_min:
        return [len(x)] + [0] * (n_bins - 1), [x_min] + [x_min] * (n_bins - 1)
    bin_size = (x_max - x_min) / n_bins
    hist = [0] * n_bins
    bin_edges = []
    for i in range(n_bins):
        bin_edges.append(x_min + i * bin_size)
    for value in x:
        if value == x_max:
            hist[n_bins - 1] += 1
            continue
        i = int((value - x_min) / bin_size)
        if i >= 0 and i < n_bins:
            hist[i] += 1
    return hist, bin_edges


In [ ]:
# Test your solution here
h,b=histogram(data,100)
print(h)

*Exercise 4:* Write a function that uses the histogram function in the previous exercise to create a text-based "graph". For example the output could look like the following:
```
[  0,  1] : ######
[  1,  2] : #####
[  2,  3] : ######
[  3,  4] : ####
[  4,  5] : ####
[  5,  6] : ######
[  6,  7] : #####
[  7,  8] : ######
[  8,  9] : ####
[  9, 10] : #####
```

Where each line corresponds to a bin and the number of `#`'s are proportional to the value of the data in the bin.

That example is only there to show the *format* — the bin ranges shown don't match our `data`,
which runs from -10 to 10. When you run it on `data` you should get ten or twenty bars of roughly
equal length, because `data` is uniform. **That flatness is the check**: if your bars are not
roughly equal, either the generator or the histogram is wrong. 

In [ ]:
def draw_histogram(x, n_bins, x_min=None, x_max=None, character="#", max_character_per_line=20):
    hist, bin_edges = histogram(x, n_bins, x_min, x_max)
    maximum = max(hist) if hist else 0
    for i in range(len(hist)):
        if maximum > 0:
            n_characters = round(hist[i] / maximum * max_character_per_line)
        else:
            n_characters = 0
        left = bin_edges[i]
        if i + 1 < len(bin_edges):
            right = bin_edges[i + 1]
        elif x_max is not None:
            right = x_max
        else:
            right = left
        print(f"[{left:7.2f}, {right:7.2f}] : {character * n_characters}")
    return hist, bin_edges


In [ ]:
# Test your solution here
draw_histogram(data,20)

## Functional Programming

*Exercise 5:* Write a function that applies a boolean function (one that returns true/false) to every element in data, and return a list of indices of elements where the result was true. Use this function to find the indices of entries greater than 0.5. 

In [ ]:
def where(mylist, myfunc):
    out = []
    for i, value in enumerate(mylist):
        if myfunc(value):
            out.append(i)
    return out


In [ ]:
positive_indices = where(data, lambda x: x > 0.5)
print("Indices of values greater than 0.5:", positive_indices[:20])
print("Number of values greater than 0.5:", len(positive_indices))


*Exercise 6:* The `in_range(mymin,mymax)` function below returns a function that tests if it's input is between the specified values. Write corresponding functions that test:
* Even
* Odd
* Greater than
* Less than
* Equal
* Divisible by

In [ ]:
def in_range(mymin,mymax):
    def testrange(x):
        return x<mymax and x>=mymin
    return testrange

# Examples:
F1=in_range(0,10)
F2=in_range(10,20)

# Test of in_range
print (F1(0), F1(1), F1(10), F1(15), F1(20))
print (F2(0), F2(1), F2(10), F2(15), F2(20))

print ("Number of Entries passing F1:", len(where(data,F1)))
print ("Number of Entries passing F2:", len(where(data,F2)))

In [ ]:
def is_even(x):
    return x % 2 == 0

def is_odd(x):
    return x % 2 != 0

def greater_than(value):
    def test(x):
        return x > value
    return test

def less_than(value):
    def test(x):
        return x < value
    return test

def equal_to(value):
    def test(x):
        return x == value
    return test

def divisible_by(value):
    def test(x):
        return x % value == 0
    return test

example = list(range(-5, 6))
print("Even indices:", where(example, is_even))
print("Odd indices:", where(example, is_odd))
print("Greater than 2:", where(example, greater_than(2)))
print("Less than 2:", where(example, less_than(2)))
print("Equal to 0:", where(example, equal_to(0)))
print("Divisible by 3:", where(example, divisible_by(3)))


In [ ]:
example = list(range(-10, 11))
print("Even count:", len(where(example, is_even)))
print("Odd count:", len(where(example, is_odd)))
print("Greater than 0 count:", len(where(example, greater_than(0))))
print("Less than 0 count:", len(where(example, less_than(0))))
print("Equal to 0 count:", len(where(example, equal_to(0))))
print("Divisible by 5 count:", len(where(example, divisible_by(5))))


*Exercise 7:* Repeat the previous exercise using `lambda` and the built-in python functions sum and map instead of your solution above. 

In [ ]:

example = list(range(-5, 6))

conditions = {
    "Even": lambda x: x % 2 == 0,
    "Odd": lambda x: x % 2 != 0,
    "Greater than 2": lambda x: x > 2,
    "Less than 2": lambda x: x < 2,
    "Equal to 0": lambda x: x == 0,
    "Divisible by 3": lambda x: x % 3 == 0
}

for name, condition in conditions.items():
    count = sum(map(condition, example))
    print(name + ":", count)


In [ ]:
for name, condition in conditions.items():
    print(name + " count:", sum(map(condition, example)))


## Monte Carlo

*Exercise 8:* Write a "generator" function called `generate_function(func,x_min,x_max,N)`, that instead of generating a flat distribution, generates a distribution with functional form coded in `func`. Note that `func` will always be > 0.  

Use the test function below and your histogramming functions above to demonstrate that your generator is working properly.

Hint: A simple, but slow, solution is to a draw random number `test_x` within the specified range and another number `p` between the `min` and `max` of the function (which you will have to determine). If `p<=function(test_x)`, then place `test_x` on the output. If not, repeat the process, drawing two new numbers. Repeat until you have the specified number of generated numbers, `N`. For this problem, it's OK to determine the `min` and `max` by numerically sampling the function.  

In [ ]:
def generate_function(func, x_min, x_max, N=1000):
    out = []
    f_max = 0.0
    n_samples = 1000
    for i in range(n_samples + 1):
        x = x_min + (x_max - x_min) * i / n_samples
        value = func(x)
        if value > f_max:
            f_max = value

    if f_max <= 0:
        return out

    while len(out) < N:
        test_x = x_min + random.random() * (x_max - x_min)
        p = random.random() * f_max
        if p <= func(test_x):
            out.append(test_x)
    return out


In [ ]:
# A test function
def test_func(x,a=1,b=1):
    return abs(a*x+b)

In [ ]:
# Test your solution here
d=generate_function(test_func,-5,5,10000)

print ("Data Length:", len(d))
print ("Minimum:", min(d), " Maximum:", max(d))

# The shape should follow test_func: a V with its point at x = -1
draw_histogram(d,20,-5,5)


*Exercise 9:* Use your function to generate 1000 numbers that are normal distributed, using the `gaussian` function below. Confirm the mean of the data is close to the `mean` you specified, and that the variance is close to `sigma**2` — note that `gaussian` takes a standard deviation, not a variance. Histogram the data. 

In [ ]:
import math

def gaussian(mean, sigma):
    def f(x):
        return math.exp(-((x-mean)**2)/(2*sigma**2))/(sigma*math.sqrt(2*math.pi))
    return f

# Example Instantiation
g1=gaussian(0,1)
g2=gaussian(10,3)

In [ ]:
# Test your solution here
import math

g1=gaussian(0,1)
d1=generate_function(g1,-5,5,1000)

print ("Mean:", mean(d1), " (should be close to 0)")
print ("Variance:", variance(d1), " (should be close to sigma**2 = 1)")
draw_histogram(d1,20,-5,5)


*Exercise 10:* Combine your `generate_function`, `where`, and `in_range` functions above to
create an integrate function.

To be precise about what it should return: **the fraction of the distribution that lies between
`x_min` and `x_max`.** Because `gaussian` is normalised (it integrates to 1 over all $x$), that
fraction *is* the integral of the distribution over that interval.

The way to get it: use `generate_function` to draw `n_points` numbers from `func`, then use
`where` and `in_range` to count how many landed between `x_min` and `x_max`. The answer is that
count divided by `n_points`.

One wrinkle — `generate_function` needs a range to generate over, and it has to be wide enough to
contain essentially the whole distribution, not just the interval you are integrating. For a
Gaussian, mean $\pm\,5\sigma$ is plenty. Feel free to add arguments to the function for that.

Use it to show that approximately 68% of a Normal distribution lies within one standard deviation
of the mean.

In [ ]:
def integrate(func, x_min, x_max, n_points=1000):
    integral_min = x_min
    integral_max = x_max

    width = x_max - x_min
    generation_min = x_min - 5 * width
    generation_max = x_max + 5 * width

    d = generate_function(func, generation_min, generation_max, n_points)
    inside = where(d, in_range(x_min, x_max))
    integral = len(inside) / n_points
    return integral


In [ ]:
# Test your solution here
g=gaussian(0,1)

print ("Within one sigma:", integrate(g,-1,1), " (should be close to 0.68)")
print ("Within two sigma:", integrate(g,-2,2), " (should be close to 0.95)")


*Exercise 11:* Your answer to Exercise 10 came from counting: you generated `N` numbers and
counted what fraction landed within one standard deviation. **Counting has an uncertainty**, so
your answer does too — and you can measure it without knowing the right answer.

Run your Exercise 10 estimate several times over (say 20) at each of `N` = 100, 1000, and 10000.
For each `N`, use your `mean` and `variance` functions from Exercise 2 on the list of estimates.

* The mean of the estimates should sit near 0.68 at every `N`.
* The *spread* should shrink as `N` grows. Work out how: is it proportional to `1/N`, or to
  `1/sqrt(N)`, or something else?

Then compare what you measured against the formula from lecture for the uncertainty on a
fraction:

$$\sigma_{\hat p} = \sqrt{\frac{p(1-p)}{N}}$$

with $p = 0.68$. At `N` = 1000 and 10000 it should agree closely. At `N` = 100 expect it to be
rougher — with only 20 trials, the spread you measure is itself a noisy estimate. Increasing
`n_trials` tightens it, which is the same effect you are studying, one level up.

In [ ]:
def estimate_spread(N, n_trials=20):
    estimates = []
    g = gaussian(0, 1)
    for i in range(n_trials):
        estimates.append(integrate(g, -1, 1, N))
    return mean(estimates), variance(estimates)



In [ ]:
# Test your solution here
import math

for N in [100,1000,10000]:
    m,v = estimate_spread(N)
    print ("N:", N, " mean of estimates:", m, " spread:", math.sqrt(v))
    print ("   formula sqrt(p(1-p)/N):", math.sqrt(0.68*0.32/N))


*Exercise 12:* Your `generate_function` works on any function you can evaluate, but it throws
draws away. When you *can* do the integral by hand, there is a method with no waste at all:
**inversion**.

The exponential distribution is

$$P(t) = \frac{1}{\tau}e^{-t/\tau} \quad (t \ge 0)$$

Its cumulative is $F(T) = 1 - e^{-T/\tau}$. Setting that equal to a uniform random number $u$ and
solving for $T$ gives

$$T = -\tau \ln(1-u)$$

**(a)** Write `generate_exp(tau,N)` that returns `N` exponentially distributed numbers using that
one line. It needs `random.random()` and `math.log` — no loop over rejected draws.

**(b)** Check it: the mean *and* the standard deviation of an exponential are both $\tau$. Use
your `mean` and `variance` functions.

**(c)** Now compare the cost. Use `generate_function` to draw the same distribution by
accept/reject over the range 0 to $5\tau$, counting how many random draws it needs per number it
keeps. Inversion needs one. How many does accept/reject need, and why that many?

In [ ]:
import math

def generate_exp(tau, N):
    out = []
    for i in range(N):
        u = random.random()
        out.append(-tau * math.log(1 - u))
    return out



In [ ]:
# Test your solution here
tau=2.
d=generate_exp(tau,100000)

print ("tau:", tau)
print ("Mean:", mean(d), " (should be close to tau)")
print ("Standard deviation:", math.sqrt(variance(d)), " (also close to tau)")

# Have a look at the shape
draw_histogram(d,20,0,5*tau)


## Percentiles

*Exercise 13:* A histogram answers "how many fell in this bin?". Turn it around and it answers
something more useful: **"below what value does 90% of the data lie?"** That value is the 90th
percentile. Every median, quartile and percentile you have seen quoted is this calculation.

Using the `histogram` function you wrote in Exercise 3:

* Take the running total down the list of bin counts — the **cumulative sum**. Entry `i` becomes
  the number of data points at or below bin `i`.
* The last entry is the total number of data points. Find where the running total first reaches
  90% of it.
* Return that bin's edge.

Write `percentile(x,fraction,n_bins=100)` that does this for any fraction, so
`percentile(d,0.5)` gives the median.

**Check it.** Generate a normal distribution with your `generate_function` and the `gaussian`
function from Exercise 9. For a normal distribution the 90th percentile is
$\mu + 1.2816\,\sigma$ — so **1.28** for `gaussian(0,1)` and **13.84** for `gaussian(10,3)`. The
median should come back near $\mu$.

Don't expect those exactly. Your answer is a bin edge, so it can only be as precise as one bin is
wide: 100 bins over a range of 10 gives a bin width of 0.1, and over a range of 30 gives 0.3.
**Landing within one bin width is a pass** — 1.3 and 14.0 are the right answers here, not near
misses.

Two things to notice, and say something about in a comment:

* Your answer is always a *bin edge*. What does that mean for how precise it can be, and what
  would you change to improve it?
* Nothing in this calculation assumed the data was normal. Why does that make percentiles a safer
  summary than a mean and a standard deviation for data you haven't looked at yet?

In [ ]:
def percentile(x, fraction, n_bins=100):
    hist, bin_edges = histogram(x, n_bins)
    target = fraction * len(x)
    cumulative = 0
    for i, count in enumerate(hist):
        cumulative += count
        if cumulative >= target:
            return bin_edges[i]
    return bin_edges[-1]



In [ ]:
# Test your solution here
g1=gaussian(0,1)
d1=generate_function(g1,-5,5,100000)

print ("median   :", percentile(d1,0.5),  " (near 0, within a bin width of 0.1)")
print ("90th pct :", percentile(d1,0.9),  " (near 1.28, so ~1.3)")

g2=gaussian(10,3)
d2=generate_function(g2,10-5*3,10+5*3,100000)

print ("median   :", percentile(d2,0.5),  " (near 10, within a bin width of 0.3)")
print ("90th pct :", percentile(d2,0.9),  " (near 13.84, so ~14.0)")
